# Rock-Paper-Scissors Model Evaluation

Loads the Teachable Machine `keras_model.h5`, runs inference on all labeled test images,
reports accuracy, and visually inspects misclassifications.

**Environment:** run this notebook using the `tm_legacy` kernel (TensorFlow 2.12.x with bundled Keras 2).
The standalone Keras 3 (`ml` env) cannot deserialize this model's nested Functional/Sequential structure.

**Expected folder layout:**
```
Model/
├── keras_model.h5
├── labels.txt
├── model.ipynb          <- this notebook
Testing/
├── rock/
│   └── *.jpg
├── paper/
│   └── *.jpg
└── scissors/
    └── *.jpg
```


## 1. Imports & Setup

In [ ]:
import os
import glob
from tensorflow.keras.models import load_model
from PIL import Image, ImageOps
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True)


## 2. Paths

In [ ]:
BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
MODEL_PATH = os.path.join(BASE_DIR, "keras_model.h5")
LABELS_PATH = os.path.join(BASE_DIR, "labels.txt")
IMAGE_FOLDER = os.path.join(BASE_DIR, "..", "Testing")

print("Model path:", MODEL_PATH)
print("Labels path:", LABELS_PATH)
print("Image folder:", IMAGE_FOLDER)


## 3. Load Model & Labels

In [ ]:
model = load_model(MODEL_PATH, compile=False)

# labels.txt lines look like "0 Rock" -> strip the index prefix
class_names = [line.strip()[2:].strip() for line in open(LABELS_PATH, "r").readlines()]
print("Classes:", class_names)


## 4. Gather Labeled Test Images

Assumes `Testing/` contains one subfolder per class (`rock/`, `paper/`, `scissors/`),
with the folder name matching a label in `labels.txt` (case-insensitive).

In [ ]:
valid_exts = (".jpg", ".jpeg", ".png", ".bmp")

samples = []  # (path, true_label)
for class_dir in sorted(os.listdir(IMAGE_FOLDER)):
    class_path = os.path.join(IMAGE_FOLDER, class_dir)
    if not os.path.isdir(class_path):
        continue
    for p in glob.glob(os.path.join(class_path, "*")):
        if p.lower().endswith(valid_exts):
            samples.append((p, class_dir))

if not samples:
    raise FileNotFoundError(f"No labeled images found under {IMAGE_FOLDER}")

print(f"Found {len(samples)} labeled images across {len(set(l for _, l in samples))} classes")


## 5. Preprocess & Run Batched Prediction

In [ ]:
size = (224, 224)
data = np.ndarray(shape=(len(samples), 224, 224, 3), dtype=np.float32)

for i, (path, _) in enumerate(samples):
    image = Image.open(path).convert("RGB")
    image = ImageOps.fit(image, size, Image.Resampling.LANCZOS)
    image_array = np.asarray(image)
    data[i] = (image_array.astype(np.float32) / 127.5) - 1

predictions = model.predict(data, verbose=0)
print("Predictions shape:", predictions.shape)


## 6. Score Predictions vs Ground Truth

In [ ]:
results = []       # (path, true_label, pred_label, confidence)
correct = 0
misclassified = []

for (path, true_label), prediction in zip(samples, predictions):
    idx = np.argmax(prediction)
    pred_label = class_names[idx]
    conf = prediction[idx]
    is_correct = pred_label.lower() == true_label.lower()

    results.append((path, true_label, pred_label, conf))
    if is_correct:
        correct += 1
    else:
        misclassified.append((path, true_label, pred_label, conf))

    marker = "OK " if is_correct else "XX "
    print(f"{marker}{os.path.basename(path):30s} true={true_label:10s} pred={pred_label:10s} ({conf:.4f})")

print(f"\nAccuracy: {correct}/{len(samples)} ({correct/len(samples):.2%})")
print(f"Misclassified: {len(misclassified)}")


## 7. Confusion Matrix

In [ ]:
labels_sorted = sorted(class_names)
n = len(labels_sorted)
label_idx = {name: i for i, name in enumerate(labels_sorted)}

cm = np.zeros((n, n), dtype=int)  # rows = true, cols = predicted
for _, true_label, pred_label, _ in results:
    cm[label_idx[true_label.capitalize()] if true_label.capitalize() in label_idx else label_idx[true_label],
       label_idx[pred_label.capitalize()] if pred_label.capitalize() in label_idx else label_idx[pred_label]] += 1

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(n)); ax.set_xticklabels(labels_sorted, rotation=45)
ax.set_yticks(range(n)); ax.set_yticklabels(labels_sorted)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix")

for i in range(n):
    for j in range(n):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")

plt.colorbar(im)
plt.tight_layout()
plt.show()


## 8. Visually Inspect Misclassifications

In [ ]:
if misclassified:
    n_mis = len(misclassified)
    cols = min(4, n_mis)
    rows = (n_mis + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, (path, true_label, pred_label, conf) in zip(axes, misclassified):
        img = Image.open(path).convert("RGB")
        ax.imshow(img)
        ax.set_title(f"True: {true_label}\nPred: {pred_label} ({conf:.2f})",
                     color="red", fontsize=10)
        ax.axis("off")

    for ax in axes[len(misclassified):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No misclassifications - model got everything right on this labeled set.")


## 9. Low-Confidence Correct Predictions (worth a second look)

In [ ]:
CONFIDENCE_THRESHOLD = 0.75

low_conf_correct = [
    (path, true_label, pred_label, conf)
    for path, true_label, pred_label, conf in results
    if conf < CONFIDENCE_THRESHOLD and pred_label.lower() == true_label.lower()
]

print(f"Correct but low-confidence (<{CONFIDENCE_THRESHOLD:.0%}): {len(low_conf_correct)}")

if low_conf_correct:
    n_low = len(low_conf_correct)
    cols = min(4, n_low)
    rows = (n_low + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1) if n_low > 1 else [axes]

    for ax, (path, true_label, pred_label, conf) in zip(axes, low_conf_correct):
        img = Image.open(path).convert("RGB")
        ax.imshow(img)
        ax.set_title(f"True: {true_label}\nPred: {pred_label} ({conf:.2f})",
                     color="orange", fontsize=10)
        ax.axis("off")

    for ax in list(axes)[len(low_conf_correct):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()
